In [36]:
# ============================================================
# Batch LLM coding of open-ended odd-one-out strategies
# Multi-Experiment Runner (Exp 1, Exp 2 & Exp 3)
# ============================================================
# Input:
#   Open_Answers/Curvature Open Answers.xlsx
#   Sheets:
#     - 'exp1 curv'    (Features: smoothness, vertices, angle, intuition, globalness, size)
#     - 'exp1 noInstr' (Features: smoothness, vertices, angle, intuition, globalness, size)
#     - 'exp2 curv'    (Features: smoothness, vertices, range, intuition, globalness, animal-like, size)
#     - 'exp2 noInstr' (Features: smoothness, vertices, range, intuition, globalness, animal-like, size)
#     - 'exp3 curv'    (Features: smoothness, vertices, range, intuition, globalness, animal-like, size)
#     - 'exp3 noInstr' (Features: smoothness, vertices, range, intuition, globalness, animal-like, size)
#
# Output:
#   Open_Answers/Curvature_Open_Answers_LLM_coded_{PROMPT_STYLE}.xlsx
#   (e.g., Curvature_Open_Answers_LLM_coded_minimal.xlsx or
#          Curvature_Open_Answers_LLM_coded_detailed.xlsx)
#   All 6 experiments are saved in separate tabs within this single workbook.
#
# Key Features:
#   - Resumable: Reads existing output file if present; skips already completed participants.
#   - Dynamic Output Filename: Automatically reflects the selected PROMPT_STYLE.
#   - Experiment-specific prompts:
#       * Exp 1 features 'angle' (difference in degrees of angle).
#       * Exp 2 & Exp 3 feature 'range' (how deep/big the curves/angles can be) and 'animal-like'.
#   - Saves progress after every batch to prevent loss.
#
# Requirements:
#   pip install -U requests pandas openpyxl python-dotenv
# ============================================================
import os
import json
import random
import time
import pandas as pd
import requests
from dotenv import load_dotenv


In [37]:
# ------------------------------------------------------------
# Verify Groq API connection and available models
# ------------------------------------------------------------
load_dotenv()
load_dotenv(".env")
load_dotenv("/Users/seoheehan/Library/CloudStorage/GoogleDrive-osooni94@gmail.com/My Drive/School/phD/Projects/Perceptual grouping/Curvature/CurvatureRespresentation/Exp1/.env")

groq_key = os.environ.get("GROQ_API_KEY")
if not groq_key:
    print("WARNING: GROQ_API_KEY not found in environment or .env. Please set it before running batch coding.")
else:
    url = "https://api.groq.com/openai/v1/models"
    try:
        r = requests.get(
            url,
            headers={"Authorization": f"Bearer {groq_key}"},
            timeout=15
        )
        print("API Status:", r.status_code)
        if r.status_code == 200:
            models = r.json().get("data", [])
            print(f"Connection successful. {len(models)} models available.")
        else:
            print("API Response:", r.text[:300])
    except Exception as e:
        print(f"Could not connect to Groq API: {e}")


API Status: 200
Connection successful. 11 models available.


In [38]:
# ------------------------------------------------------------
# 1. Setup & Experiment Configuration
# ------------------------------------------------------------
# Ensure file paths always resolve to the 'Open_Answers' folder
cwd = os.getcwd()
if os.path.basename(cwd) == "Open_Answers":
    OPEN_ANSWERS_DIR = cwd
elif os.path.exists(os.path.join(cwd, "Open_Answers")):
    OPEN_ANSWERS_DIR = os.path.join(cwd, "Open_Answers")
else:
    OPEN_ANSWERS_DIR = cwd

# Select active prompt style: 'minimal' or 'detailed'
PROMPT_STYLE = "minimal"

INPUT_FILE = os.path.join(OPEN_ANSWERS_DIR, "Curvature Open Answers.xlsx")

# Output filename dynamically reflects the chosen prompt_style
OUTPUT_FILE = os.path.join(OPEN_ANSWERS_DIR, f"Curvature_Open_Answers_LLM_coded_{PROMPT_STYLE}.xlsx")

MODEL = "openai/gpt-oss-120b"
BATCH_SIZE = 5
API_URL = "https://api.groq.com/openai/v1/chat/completions"

# Configurations for each of the 6 experiment sheets:
# - Exp 1 experiments have 'angle' as one of the features
# - Exp 2 & Exp 3 experiments have 'range' and 'animal-like' as features
# The column ordering aligns with the standard OOO analysis structure.
EXPERIMENT_CONFIGS = {
    "exp1 curv": {
        "exp_type": "exp1",
        "text_col": "strategy_open",
        "feature_cols": [
            "smoothness",
            "vertices",
            "angle",
            "intuition",
            "globalness",
            "size"
        ]
    },
    "exp1 noInstr": {
        "exp_type": "exp1",
        "text_col": "open_answer",
        "feature_cols": [
            "smoothness",
            "vertices",
            "angle",
            "intuition",
            "globalness",
            "size"
        ]
    },
    "exp2 curv": {
        "exp_type": "exp2",
        "text_col": "open_answer",
        "feature_cols": [
            "smoothness",
            "vertices",
            "range",
            "intuition",
            "globalness",
            "animal-like",
            "size"
        ]
    },
    "exp2 noInstr": {
        "exp_type": "exp2",
        "text_col": "open_answer",
        "feature_cols": [
            "smoothness",
            "vertices",
            "range",
            "intuition",
            "globalness",
            "animal-like",
            "size"
        ]
    },
    "exp3 curv": {
        "exp_type": "exp3",
        "text_col": "open_answer",
        "feature_cols": [
            "smoothness",
            "vertices",
            "range",
            "intuition",
            "globalness",
            "animal-like",
            "size"
        ]
    },
    "exp3 noInstr": {
        "exp_type": "exp3",
        "text_col": "open_answer",
        "feature_cols": [
            "smoothness",
            "vertices",
            "range",
            "intuition",
            "globalness",
            "animal-like",
            "size"
        ]
    }
}

print(f"Selected prompt style: [{PROMPT_STYLE}]")
print(f"Input file:            {INPUT_FILE}")
print(f"Output file:           {OUTPUT_FILE}")
print(f"Configured experiments: {list(EXPERIMENT_CONFIGS.keys())}")


Selected prompt style: [minimal]
Input file:            /Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature Open Answers.xlsx
Output file:           /Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature_Open_Answers_LLM_coded_minimal.xlsx
Configured experiments: ['exp1 curv', 'exp1 noInstr', 'exp2 curv', 'exp2 noInstr', 'exp3 curv', 'exp3 noInstr']


In [39]:
# ------------------------------------------------------------
# 2. LLM coding prompts (Exp 1 vs Exp 2 & Exp 3)
# ------------------------------------------------------------

PROMPT_EXP1_DETAILED = """
You are coding participants' open-ended strategies for choosing the odd-one-out among shapes.

Input is a JSON array. Each item has:
- "item_id": participant identifier
- "response": participant response

For EACH item, return one object in a JSON array under the key "results".
Each object must contain exactly:
- "item_id": same participant identifier
- "smoothness": 0 or 1
- "vertices": 0 or 1
- "angle": 0 or 1
- "intuition": 0 or 1
- "globalness": 0 or 1
- "size": 0 or 1
- "mentioned_terms": maximum 5 exact short phrases from the response
- "rationale": maximum 25 words
- "uncertain": true or false

Return ONLY valid JSON. The output must be exactly:
{"results": [ ... ]}
The value of "results" must be a JSON array.

Feature definitions & disambiguation rules:
- smoothness: Local contour quality — curvature vs. straightness, roundness vs. sharpness, pointedness, jaggedness, spikiness, wave vs. zigzag, or blunt/soft vs. sharp/hard corners, edges, points, tips, or peaks.
  * Rule: If a participant mentions "points", "corners", "edges", "peaks", or "spikes" to describe their sharpness or roundness (e.g., "sharp points", "soft or sharp corners", "straight or curved lines"), code smoothness = 1 (and vertices = 0 unless they also refer to how many there are).
- vertices: Quantity, count, number, amount, or repetition frequency of local parts — including "number of curves", "more/less curves", "how many bumps/lumps/humps/peaks/dips/waves", "number of points/corners/vertices/tips/edges/sides", "amount of turns/bends/angles/zigzags/curls", or "frequency / how many times the pattern or fluctuation repeats".
  * Rule: Code vertices = 1 ONLY when the response refers to quantity, count, number, amount, or frequency (including counting curves or angles). Mentioning "corners", "points", or "curves" without referring to their count/number is NOT vertices.
- angle: Degree, magnitude, amplitude, direction, or geometry of individual curves, angles, bumps, or bends — including difference in degrees of angle, steepness, slope, tilt, direction/orientation of a curve or bump, "size of the curve", "height of each curve/bump/peak", "depth of curves/dips", "radius or width of the curve", or "shapes of the curves and bends".
  * Rule: When participants refer to the height, depth, size, radius, or direction of an individual curve, bump, or peak (e.g., "size or direction of the curve", "height of each curve", "height of bumps/peaks"), code angle = 1.
- intuition: Gut feeling, instinct, intuition, immediate visual pop-out or first-glance salience (e.g., "what stood out to me first", "stuck out at first glance", "1st judgment", "quick glance", "whichever drew my eyes"), or holistic perceptual shortcuts such as blurring, unfocusing, or zoning out one's eyes.
- globalness: Holistic or overall shape, general shape, global configuration, outline/silhouette of the entire stimulus, or how the whole figure looks/fits together globally.
  * Rule: Do NOT code globalness = 1 just because the participant uses the word "shape" to mean the stimulus/item (e.g., "roundness of the shape" or "how many bumps each shape had"). Code globalness = 1 only when they explicitly evaluate the overall/general shape or holistic pattern.
- size: Overall physical size or scale of the entire shape/object/figure as a whole (e.g., "size of the shape/object", "bigger vs. smaller shape/picture", "overall length/area", "how much space of the box the shape took up").
  * Rule: Distinguish whole-shape size from local curve size. "Height of bumps/peaks/curves" is angle = 1 and size = 0 (unless they also mention the overall size or length of the whole shape).

Anchor examples:
- "I looked at what stood out to me first. I paid attention to the roundness of the shape, the number of curves, and the size or direction of the curve." -> smoothness=1, vertices=1 ("number of curves"), angle=1 ("size or direction of the curve"), intuition=1 ("what stood out to me first"), globalness=0, size=1.
- "I looked at the number of curves and the height of each curve." -> smoothness=0, vertices=1 ("number of curves"), angle=1 ("height of each curve"), intuition=0, globalness=0, size=0.

All codes are non-mutually exclusive. Code each feature independently: assign 1 whenever the response plausibly refers to that feature, even if other features are also mentioned.
If a response is vague or cannot be confidently mapped to a feature, set that feature to 0. Set uncertain to true only if the whole response is too vague, irrelevant, or ambiguous to code reliably.
"""

PROMPT_EXP1_MINIMAL = """
You are coding participants' open-ended strategies for choosing the odd-one-out among shapes.

Input is a JSON array. Each item has:
- "item_id": participant identifier
- "response": participant response

For EACH item, return one object in a JSON array under the key "results".
Each object must contain exactly:
- "item_id": same participant identifier
- "smoothness": 0 or 1
- "vertices": 0 or 1
- "angle": 0 or 1
- "intuition": 0 or 1
- "globalness": 0 or 1
- "size": 0 or 1
- "mentioned_terms": maximum 5 exact short phrases from the response
- "rationale": maximum 25 words
- "uncertain": true or false

Return ONLY valid JSON. The output must be exactly:
{"results": [ ... ]}
The value of "results" must be a JSON array.

Feature definitions:
- smoothness: If the explanation is related to sharpness or smoothness / curvature
- vertices: If the explanation is related to the count of vertices, points, corners, or edges
- angle: If the explanation is related to angle (difference in degrees of angle, steepness, slope)
- intuition: Gut feeling, instinct, or intuition
- globalness: Overall shape or global configuration
- size: Size of the shape (largeness, smallness, extent)

All codes are non-mutually exclusive. Code each feature independently: assign 1 whenever the response plausibly refers to that feature.
If a response is vague or cannot be confidently mapped to a feature, set that feature to 0. Set uncertain to true only if the whole response is too vague, irrelevant, or ambiguous to code reliably.
"""

PROMPT_EXP2_DETAILED = """
You are coding participants' open-ended strategies for choosing the odd-one-out among shapes.

Input is a JSON array. Each item has:
- "item_id": participant identifier
- "response": participant response

For EACH item, return one object in a JSON array under the key "results".
Each object must contain exactly:
- "item_id": same participant identifier
- "smoothness": 0 or 1
- "vertices": 0 or 1
- "range": 0 or 1
- "intuition": 0 or 1
- "globalness": 0 or 1
- "animal-like": 0 or 1
- "size": 0 or 1
- "mentioned_terms": maximum 5 exact short phrases from the response
- "rationale": maximum 25 words
- "uncertain": true or false

Return ONLY valid JSON. The output must be exactly:
{"results": [ ... ]}
The value of "results" must be a JSON array.

Feature definitions & disambiguation rules:
- smoothness: Local edge/corner quality — sharpness vs. smoothness/roundness of the edges, corners, points, tips, or vertices (e.g., pointedness, jaggedness, spikiness, or blunt/soft/rounded vs. sharp/hard corners or edges).
  * Rule 1: If a participant mentions "points", "corners", "edges", or "spikes" to describe their local sharpness or roundness (e.g., "sharp points", "blunt or sharp edges", "rounded or sharp corners"), code smoothness = 1 (and vertices = 0 unless they also refer to how many there are).
  * Rule 2 (smoothness vs. range): If "smoothness" or "round" refers to the local smoothness/roundness of an edge or corner, code smoothness = 1. However, if "smoothness" refers to the smoothness of a whole line/contour vs. stronger/deeper curves in a line, OR if "round"/"roundedness"/"circular" refers to the overall roundedness of the shape (vs. deep/pronounced curves or indentations), code range = 1.
- vertices: Quantity, count, number, amount, or repetition frequency of local parts — including "number of curves", "more/less curves", "how many bumps/lumps/peaks/dips/spikes/grooves/protrusions", "number of points/corners/vertices/tips/edges/sides", "amount of turns/bends/angles", or "frequency of peaks".
  * Rule: Code vertices = 1 ONLY when the response refers to quantity, count, number, amount, or frequency (including "number of curves"). Mentioning "points", "edges", "corners", or "spikes" without referring to their count/number is NOT vertices.
- range: How deep, big, strong, tall, intense, or pronounced the curves, angles, peaks, spikes, indentations, or grooves are (amplitude, depth, height, or intensity of curves/angles). Depending on context, range also includes:
  * Overall roundedness or circularity of the shape (e.g., "how circular", overall roundness of the shape vs. deep curves/protrusions).
  * Smoothness of a whole line/contour vs. stronger or deeper curves in a line (mild/gentle contour vs. strong/deep curves).
- intuition: Gut feeling, instinct, intuition, immediate visual pop-out or first-glance salience (e.g., "what stood out to me first", "stuck out at first glance", "first judgment", "quick glance", "whichever drew my eyes"), or holistic perceptual shortcuts such as blurring, unfocusing, or zoning out one's eyes.
- globalness: Holistic or overall shape, general shape, global configuration, outline/silhouette of the entire stimulus, symmetry, or how the whole figure looks/fits together globally.
  * Rule: Do NOT code globalness = 1 just because the participant uses the word "shape" to mean the stimulus/item (e.g., "sharpness of the shape" or "whether the shape had round edges"). Code globalness = 1 only when they evaluate the overall/general shape, outline, or global structure.
- animal-like: When participants mention that certain shapes look like animals, specific animals (e.g., dog, wolf), animal heads/faces/parts, creatures, or organisms.
- size: Overall physical size or scale of the entire shape/object/figure as a whole (e.g., "size of the shape/object", "bigger vs. smaller shape", "overall area").

Anchor examples:
- "Sharp Edges and how circular" -> smoothness=1 ("Sharp Edges" = local edge sharpness), range=1 ("how circular" = overall roundedness of the shape).
- "the intensity and frequency of peaks, and whether or not it was curvy versus sharp" -> smoothness=1 ("curvy versus sharp"), vertices=1 ("frequency of peaks"), range=1 ("intensity of peaks").

All codes are non-mutually exclusive. Code each feature independently: assign 1 whenever the response plausibly refers to that feature, even if other features are also mentioned.
If a response is vague or cannot be confidently mapped to a feature, set that feature to 0. Set uncertain to true only if the whole response is too vague, irrelevant, or ambiguous to code reliably.
"""

PROMPT_EXP2_MINIMAL = """
You are coding participants' open-ended strategies for choosing the odd-one-out among shapes.

Input is a JSON array. Each item has:
- "item_id": participant identifier
- "response": participant response

For EACH item, return one object in a JSON array under the key "results".
Each object must contain exactly:
- "item_id": same participant identifier
- "smoothness": 0 or 1
- "vertices": 0 or 1
- "range": 0 or 1
- "intuition": 0 or 1
- "globalness": 0 or 1
- "animal-like": 0 or 1
- "size": 0 or 1
- "mentioned_terms": maximum 5 exact short phrases from the response
- "rationale": maximum 25 words
- "uncertain": true or false

Return ONLY valid JSON. The output must be exactly:
{"results": [ ... ]}
The value of "results" must be a JSON array.

Feature definitions:
- smoothness: If the explanation is related to sharpness or smoothness / curvature
- vertices: If the explanation is related to the count of vertices, points, corners, or edges
- range: If the explanation is related to range (how deep or big the curves or angles can be)
- intuition: Gut feeling, instinct, or intuition
- globalness: Overall shape or global configuration
- animal-like: When people mention that certain shapes look like animals or creatures
- size: Size of the shape (largeness, smallness, extent)

All codes are non-mutually exclusive. Code each feature independently: assign 1 whenever the response plausibly refers to that feature.
If a response is vague or cannot be confidently mapped to a feature, set that feature to 0. Set uncertain to true only if the whole response is too vague, irrelevant, or ambiguous to code reliably.
"""

# You can also change the prompt style here; doing so automatically syncs OUTPUT_FILE
def set_prompt_style(style: str):
    """Update active prompt style and dynamically refresh OUTPUT_FILE path."""
    global PROMPT_STYLE, OUTPUT_FILE
    if style not in ("minimal", "detailed"):
        print(f"Note: Using custom prompt style '{style}'.")
    PROMPT_STYLE = style
    OUTPUT_FILE = os.path.join(OPEN_ANSWERS_DIR, f"Curvature_Open_Answers_LLM_coded_{PROMPT_STYLE}.xlsx")
    print(f"Prompt style set to: [{PROMPT_STYLE}]")
    print(f"Output file set to:   [{OUTPUT_FILE}]")

# Initialize or re-sync with current PROMPT_STYLE
set_prompt_style(PROMPT_STYLE)

def get_prompt(exp_type: str, style: str = None) -> str:
    """Return the prompt corresponding to experiment type and style."""
    active_style = style if style is not None else PROMPT_STYLE
    if exp_type == "exp1":
        return PROMPT_EXP1_DETAILED if active_style == "detailed" else PROMPT_EXP1_MINIMAL
    elif exp_type in ("exp2", "exp3"):
        return PROMPT_EXP2_DETAILED if active_style == "detailed" else PROMPT_EXP2_MINIMAL
    else:
        raise ValueError(f"Unknown experiment type: {exp_type}")


Prompt style set to: [minimal]
Output file set to:   [/Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature_Open_Answers_LLM_coded_minimal.xlsx]


In [40]:
# ------------------------------------------------------------
# 3. Dynamic JSON schema generator for structured output
# ------------------------------------------------------------
def get_response_schema(feature_cols: list[str]) -> dict:
    """Generate a strict JSON schema for validating LLM structured output."""
    props = {col: {"type": "integer", "enum": [0, 1]} for col in feature_cols}
    props.update({
        "item_id": {"type": "string"},
        "mentioned_terms": {"type": "array", "items": {"type": "string"}},
        "rationale": {"type": "string"},
        "uncertain": {"type": "boolean"}
    })
    return {
        "type": "object",
        "properties": {
            "results": {
                "type": "array",
                "items": {
                    "type": "object",
                    "properties": props,
                    "required": ["item_id", *feature_cols, "mentioned_terms", "rationale", "uncertain"]
                }
            }
        },
        "required": ["results"]
    }


In [41]:
# ------------------------------------------------------------
# 4. Helper functions for batch LLM coding
# ------------------------------------------------------------
def make_error_row(feature_cols: list[str], error_message: str) -> dict:
    """Create a standardized failed-coding row."""
    return {
        **{feat: None for feat in feature_cols},
        "mentioned_terms": [],
        "rationale": f"API error: {error_message}",
        "uncertain": True
    }


def code_batch(
    batch_df: pd.DataFrame,
    text_col: str,
    feature_cols: list[str],
    prompt: str,
    max_attempts: int = 4
) -> list[dict]:
    """
    Code a batch of participant responses in one Groq API request.
    Handles empty responses cleanly without wasting API calls.
    """
    coded_rows = []
    non_empty_indices = []

    for idx, row in batch_df.iterrows():
        resp_text = str(row[text_col]).strip() if pd.notna(row[text_col]) else ""
        if not resp_text or resp_text.lower() in ("nan", "none", ""):
            # Participant left response blank
            coded_rows.append({
                "subj_id": str(row["subj_id"]),
                text_col: "",
                **{feat: 0 for feat in feature_cols},
                "mentioned_terms": [],
                "rationale": "No response provided",
                "uncertain": True
            })
        else:
            non_empty_indices.append(idx)

    if not non_empty_indices:
        return coded_rows

    active_batch = batch_df.loc[non_empty_indices]

    items = [
        {
            "item_id": str(row["subj_id"]),
            "response": str(row[text_col]).strip()
        }
        for _, row in active_batch.iterrows()
    ]

    expected_ids = {str(x) for x in active_batch["subj_id"]}
    payload_text = json.dumps(items, ensure_ascii=False)

    api_key = os.environ.get("GROQ_API_KEY", "")
    if not api_key:
        raise ValueError("GROQ_API_KEY environment variable is not set.")

    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }

    request_body = {
        "model": MODEL,
        "temperature": 0,
        "max_completion_tokens": 8000,
        "response_format": {"type": "json_object"},
        "messages": [
            {"role": "system", "content": prompt},
            {"role": "user", "content": payload_text}
        ]
    }

    last_error = None

    for attempt in range(1, max_attempts + 1):
        print(f"Groq attempt {attempt}/{max_attempts}...", flush=True)

        try:
            response = requests.post(
                API_URL,
                headers=headers,
                json=request_body,
                timeout=120
            )
            response.raise_for_status()

            content = response.json()["choices"][0]["message"]["content"]
            parsed = json.loads(content)

            # Accept {"results": [...]}, direct [...], or single {...}
            if isinstance(parsed, dict) and "results" in parsed:
                results = parsed["results"]
            elif isinstance(parsed, list):
                results = parsed
            elif isinstance(parsed, dict) and "item_id" in parsed:
                results = [parsed]
            else:
                raise ValueError(
                    f"Model returned unexpected JSON structure: {parsed!r}"
                )

            if not isinstance(results, list):
                raise ValueError("Model did not return a list of coded results.")

            returned_ids = {str(r.get("item_id")) for r in results}
            if returned_ids != expected_ids:
                raise ValueError(
                    f"ID mismatch. Expected: {sorted(expected_ids)}, Received: {sorted(returned_ids)}"
                )

            for result in results:
                # Validate feature codes
                for feature in feature_cols:
                    value = result.get(feature)
                    if type(value) is not int or value not in (0, 1):
                        raise ValueError(
                            f"Invalid code for {feature} in item {result.get('item_id')}: {value!r}"
                        )

                if type(result.get("uncertain")) is not bool:
                    raise ValueError("uncertain must be a boolean.")
                if not isinstance(result.get("mentioned_terms"), list):
                    raise ValueError("mentioned_terms must be a list.")
                if not isinstance(result.get("rationale"), str):
                    raise ValueError("rationale must be a string.")

                original = active_batch[
                    active_batch["subj_id"].astype(str) == str(result["item_id"])
                ].iloc[0]

                coded_rows.append({
                    "subj_id": str(original["subj_id"]),
                    text_col: original[text_col],
                    **{col: result[col] for col in feature_cols},
                    "mentioned_terms": "; ".join(result["mentioned_terms"]) if isinstance(result["mentioned_terms"], list) else str(result["mentioned_terms"]),
                    "rationale": result["rationale"],
                    "uncertain": result["uncertain"]
                })

            return coded_rows

        except (requests.RequestException, json.JSONDecodeError, ValueError, KeyError) as exc:
            last_error = exc
            print(f"Attempt {attempt} failed: {repr(exc)}", flush=True)

            if attempt == max_attempts:
                raise

            wait_seconds = min(15 * (2 ** (attempt - 1)), 240) + random.uniform(0, 5)
            print(f"Waiting {wait_seconds:.0f} seconds...", flush=True)
            time.sleep(wait_seconds)

    raise last_error


# ------------------------------------------------------------
# 4b. Word-cloud term extraction, cleaning & feature categorization
# ------------------------------------------------------------
import re

TYPO_FIXES = [
    (r"\bbumbs\b", "bumps"),
    (r"\bcruve\b", "curve"),
    (r"\bcuve\b", "curve"),
    (r"\bshap\b", "shape"),
    (r"\bspiker\b", "spikier"),
    (r"\bawreness\b", "awareness"),
    (r"\boverthik\b", "overthink"),
    (r"\bdesigna\b", "design"),
    (r"\bmanu\b", "many"),
    (r"\bhow any bumps\b", "how many bumps"),
    (r"\bhow much vertices\b", "how many vertices"),
    (r"\btoo much points\b", "how many points"),
    (r"\bnumbers of\b", "number of"),
    (r"\bnumber of angle\b", "number of angles"),
    (r"\bnumber of the peaks\b", "number of peaks"),
    (r"similar‑sloped", "similar-sloped"),
]

FILLER_EXACT = {
    "[]", "", "paid attention", "three criteria", "all 3", "three options",
    "each of the shapes", "with the others", "trying to compare",
    "identifying whether", "picked the one", "others", "shape is",
    "each shape", "the shape", "focusing on", "group those",
    "prioritizing that", "repeated to myself", "which doesn't belong",
    "odd one out", "2 with edges", "press curved", "press the one with edges",
    "final image had two", "two identical ones", "analyze shape",
    "judged it based on the looks", "specific details",
}

ANIMAL_PATTERNS = [
    r"\banimals?\b", r"\bdogs?\b", r"\bwolf\b", r"\bhuman\b",
    r"resembled the head", r"associations with figures", r"apophenia",
]

COUNT_NOUN_MAP = [
    (r"\bbumps?\b", "number of bumps"),
    (r"\bhumps?\b", "number of humps"),
    (r"\blumps?\b", "number of lumps"),
    (r"\bcurves?\b|\bcurvatures?\b", "number of curves"),
    (r"\bcurls?\b", "number of curls"),
    (r"\bloops?\b", "number of loops"),
    (r"\bpeaks?\b", "number of peaks"),
    (r"\bdips?\b", "number of dips"),
    (r"\bpoints?\b", "number of points"),
    (r"\bvertices\b|\bvertex\b", "number of vertices"),
    (r"\bcorners?\b", "number of corners"),
    (r"\bspikes?\b|\bthorns?\b", "number of spikes"),
    (r"\bprotrusions?\b|\bprotruding\b", "number of protrusions"),
    (r"\bgrooves?\b", "number of grooves"),
    (r"\bedges?\b", "number of edges"),
    (r"\bsides?\b", "number of sides"),
    (r"\bangles?\b", "number of angles"),
    (r"\bturns?\b", "number of turns"),
    (r"\bbends?\b", "number of bends"),
    (r"\bzigzags?\b", "number of zigzags"),
    (r"\bwaves?\b", "number of waves"),
    (r"\bfluctuations?\b", "number of fluctuations"),
    (r"\blines?\b", "number of lines"),
]

QUANTITY_CUE = re.compile(
    r"(\bnumber of\b|\bamount of\b|\bhow many\b|#\s*of\b|\bcount(?:ing)?\b|"
    r"\bmore\b|\bless\b|\bfewer\b|\bmost\b|\blimited\b|\ba lot of\b|"
    r"\btwo\b|\bthree\b|\bsame number\b|\bsame amount\b|\bdifferent numbers?\b|"
    r"\bdifferent amount\b|\brepetitions?\b)"
)


def split_compound_phrase(phrase: str) -> list[str]:
    """Split contrast or compound phrases into separate tokens while preserving context nouns."""
    p = phrase.strip()
    if not p:
        return []

    if re.search(r"number and size of (?:the )?curves?", p):
        return ["number of curves", "size of curve"]
    if re.search(r"size or direction of (?:the )?curves?", p):
        return ["size of curve", "direction of curve"]
    if re.search(r"intensity and frequency of peaks", p):
        return ["intensity of peaks", "frequency of peaks"]
    if re.search(r"steepness or angle of (?:the )?bump", p):
        return ["steepness", "angle"]
    if re.search(r"general shape/size", p):
        return ["general shape", "size"]
    if re.search(r"size and space", p):
        return ["size"]
    if re.search(r"shape, smoothness, or patterns", p):
        return ["overall shape", "smoothness"]
    if re.search(r"moving/sloping/angling", p):
        return ["slope", "angle", "direction"]

    m_shared_noun = re.match(
        r"^(?:whether (?:they|it) (?:were|was|had) )?"
        r"(sharp|sharper|soft|softer|round|rounded|blunt|pointy|pointed|curved|curvy|straight|jagged|flat|smooth|smoother|hard|harsh|ridged|obtuse)"
        r"\s*(?:vs\.?|versus|or|and|/|&)\s*"
        r"(sharp|sharper|soft|softer|round|rounded|blunt|pointy|pointed|curved|curvy|straight|jagged|flat|smooth|smoother|hard|harsh|ridged|obtuse)"
        r"(?:\s+(edges?|corners?|lines?|points?|peaks?|angles?|curves?|outlines?|shapes?))?$",
        p
    )
    if m_shared_noun:
        adj1, adj2, noun = m_shared_noun.group(1), m_shared_noun.group(2), m_shared_noun.group(3)
        if noun:
            return [f"{adj1} {noun}", f"{adj2} {noun}"]
        return [adj1, adj2]

    m_lead_noun = re.match(
        r"^(edges?|corners?|lines?|points?|peaks?|angles?|curves?|outlines?)\s+"
        r"(?:were|are|was)?\s*"
        r"(sharp|sharper|soft|softer|round|rounded|blunt|pointy|pointed|curved|curvy|straight|jagged|flat|smooth|smoother|hard|harsh)"
        r"\s*(?:vs\.?|versus|or|and|/)\s*"
        r"(sharp|sharper|soft|softer|round|rounded|blunt|pointy|pointed|curved|curvy|straight|jagged|flat|smooth|smoother|hard|harsh)$",
        p
    )
    if m_lead_noun:
        noun, adj1, adj2 = m_lead_noun.group(1), m_lead_noun.group(2), m_lead_noun.group(3)
        return [f"{adj1} {noun}", f"{adj2} {noun}"]

    if re.search(r"\bless/more curves\b", p):
        return ["number of curves"]
    if re.search(r"\bpointy\s*&\s*non\s*pointy\b", p):
        return ["pointy"]
    if re.search(r"\bround or square or pointy\b", p):
        return ["round", "square", "pointy"]
    if re.search(r"\bmore curvy or rigid\b", p):
        return ["curvy", "rigid"]
    if re.search(r"\bmore square or rounded\b", p):
        return ["square", "rounded"]
    if re.search(r"\bspikes or if it was smooth\b", p):
        return ["spikes", "smooth"]
    if re.search(r"\bcurves or ridged lines\b", p):
        return ["curves", "ridged lines"]
    if re.search(r"\bcurves/spikes\b", p):
        return ["curves", "spikes"]
    if re.search(r"\bpoints/waves\b", p):
        return ["points", "waves"]
    if re.search(r"\bcurves or vertices\b", p):
        return ["curves", "vertices"]
    if re.search(r"\bnumber of bumps/points\b", p):
        return ["number of bumps", "number of points"]

    if re.search(r"\s+(?:vs\.?|versus)\s+", p):
        parts = re.split(r"\s+(?:vs\.?|versus)\s+", p)
        return [pt.strip() for pt in parts if pt.strip()]

    return [p]


def normalize_single_token(token: str, exp_type: str, row_range_coded: int = 0) -> tuple[str, str]:
    """
    Map a single extracted phrase to (wordcloud_term, feature_category).
    feature_category is restricted to:
      - Exp 1: 'smoothness', 'number of vertices', 'angle', or ''
      - Exp 2 & 3: 'smoothness', 'number of vertices', 'range', or ''
    """
    t = token.strip().lower()
    t = re.sub(r"^['\"\(\[\{]+|['\"\)\]\}]+$", "", t).strip()
    if not t or t in FILLER_EXACT:
        return ("", "")

    # 1. Animal-like -> group all as 'animal-like' (category left blank per instructions)
    for pat in ANIMAL_PATTERNS:
        if re.search(pat, t):
            return ("animal-like", "")

    # 2. Quantity / Number of vertices (preserving specific noun vocabulary)
    if "frequency of peaks" in t:
        return ("frequency of peaks", "number of vertices")
    if "frequency of lines" in t or "how many times the pattern" in t or "frequency of the pictures" in t:
        return ("frequency", "number of vertices")
    if t in ("vertices", "vertex", "numbers", "how many"):
        return ("number of vertices", "number of vertices")

    if QUANTITY_CUE.search(t):
        for noun_pat, canonical_count in COUNT_NOUN_MAP:
            if re.search(noun_pat, t):
                return (canonical_count, "number of vertices")

    # 3. Exp 1 Angle vs Exp 2/3 Range
    third_feat = "angle" if exp_type == "exp1" else "range"

    if re.search(r"\bheight of (?:each )?(?:curves?|bumps?|peaks?|spikes?|the others)\b|\bsignificantly taller\b", t):
        if "spike" in t:
            return ("height of spikes", third_feat)
        elif "peak" in t:
            return ("height of peaks", third_feat)
        elif "bump" in t:
            return ("height of bumps", third_feat)
        elif "curve" in t:
            return ("height of curve", third_feat)
        return ("height", third_feat)

    if re.search(r"\bintensity of peaks\b", t):
        return ("intensity of peaks", third_feat)
    if re.search(r"\bdepth of dips\b|\bdepth\b", t):
        return ("depth", third_feat)
    if re.search(r"\bradius of (?:the )?curve\b|\bradius\b", t):
        return ("radius", third_feat)
    if re.search(r"\bsize of (?:the )?curves?\b|\blarge curves?\b|\bone large bump\b", t):
        return ("size of curve", third_feat)
    if re.search(r"\bshapes? of (?:the )?curves?\b|\bshape/bend of the line\b|\bcurve of one shape\b", t):
        return ("shape of curve", third_feat)
    if re.search(r"\bdirection of (?:the )?(?:curves?|bumps?)\b", t):
        return ("direction of curve", third_feat)
    if re.search(r"\bradial outline\b|\bradial regularity\b", t):
        return ("radial outline", third_feat)
    if re.search(r"\bless protruding\b|\bshapes reached out\b", t) and exp_type in ("exp2", "exp3"):
        return ("protrusion depth", "range")

    if re.search(r"\bsteepness\b|\bvery steep\b|\bsteep\b|\bsteeper\b", t):
        return ("steepness", third_feat)
    if re.search(r"\bslopes?\b|\bsimilar-sloped\b|\bsloping\b", t):
        return ("slope", third_feat)
    if re.search(r"\bangularity\b", t):
        return ("angularity", third_feat)
    if re.search(r"\bobtuse\b", t):
        return ("obtuse", third_feat)
    if re.search(r"\bangles over 180 degrees\b|\bdegree of angle\b", t):
        return ("degree of angle", third_feat)

    if t in ("angle", "angles", "the angle", "looking for angles"):
        if exp_type == "exp1":
            return ("angle", "angle")
        else:
            return ("angles", "smoothness")

    if re.search(r"\bhow circular\b|\bcircular\b", t):
        return ("circular", "range" if exp_type in ("exp2", "exp3") else "smoothness")
    if exp_type in ("exp2", "exp3") and re.search(r"\bhow round the shape\b|\brounder shapes\b|\bbig/round\b", t):
        return ("roundness", "range")
    if exp_type in ("exp2", "exp3") and row_range_coded == 1 and re.search(r"\broundness\b|\broundedness\b|\brounder\b|\bhow round\b", t):
        return ("roundness", "range")

    # 4. Smoothness / Sharpness / Roundness / Curvature vocabulary
    SMOOTHNESS_RULES = [
        (r"\bsharp edges?\b|\bsharpest edge\b|\bedges are sharp\b|\bless sharp edges\b|\bedge sharpness\b|\bsharp the edges\b", "sharp edges"),
        (r"\bsharp corners?\b|\bcorners were sharp\b|\bhard corner(?: line)?\b|\bsharpness of corners\b", "sharp corners"),
        (r"\bsharp points?\b|\bsharper points?\b|\bsharp or rounded point\b", "sharp points"),
        (r"\bsharp angles?\b|\bsharpness of angles\b", "sharp angles"),
        (r"\bsharp lines?\b|\bsharp lined\b|\bsharper lines?\b|\bsharpness of lines\b", "sharp lines"),
        (r"\bsharp curves?\b|\bsharp the curves\b|\bsharpness of (?:the )?curves\b|\bcurve sharpness\b", "sharp curves"),
        (r"\bsharpness of vertices\b", "sharp vertices"),
        (r"\bsharpness\b|\bhow sharp\b|\bappear sharp\b|\bsharp\b|\bsharper\b|\bextremity \(eg\. sharpness\)", "sharpness"),
        (r"\bpointy edges?\b|\bmore pointy edges\b", "pointy edges"),
        (r"\bpointy corners?\b", "pointy corners"),
        (r"\bpointy peaks?\b|\bpeaks were really pointy\b|\bsharp/curvy the peaks\b", "pointy peaks"),
        (r"\bpointiness\b|\bpointyness\b", "pointiness"),
        (r"\bpointy\b|\bpointier\b|\bpointed\b", "pointy"),
        (r"\bsmooth edges?\b|\bedges are smooth\b|\bsofter edges\b|\bsoft edges\b", "smooth edges"),
        (r"\bsmooth corners?\b|\bsoft corners?\b", "soft corners"),
        (r"\bsmooth curves?\b", "smooth curves"),
        (r"\bsmooth lines?\b|\bsmooth the lines\b|\bsmooth, softer lines\b", "smooth lines"),
        (r"\bsmooth points?\b|\bsmoother points\b", "smooth points"),
        (r"\bsmoothness\b|\bamount of smoothness\b|\bsmooth\b|\bsmoother\b|\bmore smooth\b|\bsmooth areas\b", "smoothness"),
        (r"\brounded edges?\b|\bround edges?\b", "rounded edges"),
        (r"\brounded corners?\b|\bround corners?\b|\bcorners were rounded\b|\bmore round around the corners\b", "rounded corners"),
        (r"\brounded angles?\b|\bhard or soft angles\b", "rounded angles"),
        (r"\broundness\b|\broundedness\b|\broundness of the shape\b", "roundness"),
        (r"\brounded\b|\bmore rounded\b|\bhow rounded\b|\brounded ones\b", "rounded"),
        (r"\bround\b|\brounder\b|\bmost round\b|\bhow round(?: the shape)?\b|\brounder shapes\b|\bwhich were round\b", "round"),
        (r"\bcurved edges?\b", "curved edges"),
        (r"\bcurved lines?\b|\bcurvy lines?\b|\bcurvy the lines\b|\bcurviness of the lines\b", "curvy lines"),
        (r"\bcurviness\b", "curviness"),
        (r"\bcurvature\b|\bcurvature difference\b|\bcurvature similar\b|\bchecking curvature\b", "curvature"),
        (r"\bcurvy\b|\bequally curvy\b", "curvy"),
        (r"\bcurved\b|\btwo curved\b|\bline began to curve\b", "curved"),
        (r"\bcurves\b|\bdifferent curve\b|\bonly curve\b|\bcurve of the shapes\b|\bcurves of the shapes\b|\bsame type of curves\b|\bcurve\b|\bends of each curve\b", "curves"),
        (r"\bstraight edges?\b", "straight edges"),
        (r"\bstraight lines?\b|\bstraight part\b|\blinear line differences\b", "straight lines"),
        (r"\bstraight\b", "straight"),
        (r"\bspikiness\b", "spikiness"),
        (r"\bspiky\b|\bspikier\b|\bmost spiky\b", "spiky"),
        (r"\bspikes\b|\bsharp thorns\b", "spikes"),
        (r"\bjagged(?: edges)?\b", "jagged"),
        (r"\bzigzags?\b", "zigzag"),
        (r"\bwaves?\b|\bwave pattern\b|\bwavelengths\b", "wave"),
        (r"\bblunt(?: edges)?\b", "blunt"),
        (r"\bdull\b", "dull"),
        (r"\bsoftness\b|\bsoftness or hardness of the corners\b", "softness"),
        (r"\bsoft\b", "soft"),
        (r"\bhard\b|\bharsh lines\b|\bharsh\b", "harshness"),
        (r"\bfluidity\b", "fluidity"),
        (r"\brigidity(?: of the corners)?\b|\brigid\b", "rigidity"),
        (r"\bridged(?: lines)?\b", "ridged"),
        (r"\bflat edges?\b|\bflat surfaces?\b", "flat edges"),
        (r"\bblocky\b", "blocky"),
        (r"\bsquare\b|\bsquared\b|\bcircles or squares\b", "square"),
        (r"\bblobby\b", "blobby"),
        (r"\btriangle[\s-]like\b", "triangle-like"),
        (r"\bclearly defined\b", "defined"),
        (r"\bcorners?\b|\bonly corner\b|\bshapes' corners\b", "corners"),
        (r"\bedges?\b|\bsimilar edges\b", "edges"),
        (r"\bpeaks?\b", "peaks"),
        (r"\btips?\b", "tips"),
        (r"\bpoints?\b|\bslightly different points\b", "points"),
        (r"\bbends?\b", "bends"),
        (r"\bturns were smooth\b|\bturns\b", "turns"),
        (r"\bgrooves?\b", "grooves"),
        (r"\bprotrusions?\b|\btype of protrusion\b", "protrusions"),
    ]
    for pat, norm_word in SMOOTHNESS_RULES:
        if re.search(pat, t):
            return (norm_word, "smoothness")

    # 5. Non-core features (kept in wordcloud_terms, left blank '' in feature category)
    OTHER_RULES = [
        (r"\boverall shapes?\b|\boverall shape of the line\b|\boverall picture\b|\boverall silhouette\b|\boverall form/shape\b|\bbigger picture\b|\bbig picture\b", "overall shape"),
        (r"\bgeneral shapes?\b|\bbase shape\b|\bshape of the blocks\b", "general shape"),
        (r"\boutlines?\b|\boutline of the shape\b", "outline"),
        (r"\bsimilar shapes?\b|\bshape similarit(?:y|ies)\b|\bsimilarity in the shapes\b|\bstructural similarities\b|\bsimilarities\b|\bsimilarity\b|\bmost similar\b|\bleast similar\b|\bless similar\b|\blooked more similar\b", "shape similarity"),
        (r"\bsymmetry\b|\bcentral symmetry\b", "symmetry"),
        (r"\borientations?\b|\bdirection\b|\brotation\b|\btilted\b|\bleaning left/right\b|\bimage was flipped\b|\bfaced the same direction\b|\bfacing the same way\b|\boriented differently\b", "orientation"),
        (r"\bposition(?:ing)?\b|\blocation of curves\b|\bwhere it's located\b|\bdip at a certain location\b", "position"),
        (r"\bwhat stood out(?: to me first)?\b|\bstood out\b|\bstuck out(?: at first glance)?\b|\binstinctively stood out\b|\bcaught my attention\b|\bdrew my eyes\b|\boddest one first\b|\bmost out of place\b", "stood out"),
        (r"\bfirst glance\b|\bquick glance\b|\bglancing quickly\b|\b1st judgement\b|\bfirst impression\b|\binitial observation\b|\bnoticed immediately\b|\binstantly saw\b|\beyeballed\b", "first glance"),
        (r"\bintuition\b|\btrusted my intuition\b|\bgroup it intuitively\b|\binstinctual\b|\bfirst instinct\b|\bmy gut\b|\bvibes based\b", "intuition"),
        (r"\bunfocus(?: my)? eyes\b|\bzone out\b|\bclose my eyes\b|\bfarther away\b", "unfocus eyes"),
        (r"\bsizes?\b|\bsize of the shapes?\b|\bsize of the object\b|\boverall shape size\b|\bsmaller or bigger size\b|\bsmaller vs wider\b|\bskinny/narrow\b|\bhow wide\b|\barea\b|\blength\b|\btook up space\b|\bfilling the entire box\b|\bbig/tall/small\b|\bbig\b|\bsmall(?:er)?\b", "size"),
    ]
    for pat, norm_word in OTHER_RULES:
        if re.search(pat, t):
            return (norm_word, "")

    return (t, "")


def extract_wordcloud_columns(mentioned_terms_val, exp_type: str, row_range_coded: int = 0) -> tuple[str, str]:
    """Extract cleaned wordcloud_terms and 1-to-1 aligned wordcloud_features from mentioned_terms."""
    if pd.isna(mentioned_terms_val):
        return ("", "")
    raw = str(mentioned_terms_val).strip()
    if not raw or raw == "[]":
        return ("", "")

    for pat, repl in TYPO_FIXES:
        raw = re.sub(pat, repl, raw, flags=re.IGNORECASE)

    raw_parts = [p.strip() for p in raw.split(";") if p.strip()]
    atomic_phrases = []
    for p in raw_parts:
        atomic_phrases.extend(split_compound_phrase(p))

    pairs = []
    seen_terms = set()
    for p in atomic_phrases:
        term, cat = normalize_single_token(p, exp_type, row_range_coded)
        if not term:
            continue
        if term not in seen_terms:
            seen_terms.add(term)
            pairs.append((term, cat))

    cat_pairs = [(t, c) for t, c in pairs if c]
    uncat_pairs = [(t, c) for t, c in pairs if not c]
    ordered = cat_pairs + uncat_pairs

    terms_str = "; ".join(t for t, _ in ordered)
    cats_str = "; ".join(c for _, c in cat_pairs)
    return (terms_str, cats_str)


def add_wordcloud_columns_to_df(df: pd.DataFrame, exp_type: str) -> pd.DataFrame:
    """Add or refresh 'wordcloud_terms' and 'wordcloud_features' columns on a sheet DataFrame."""
    if "mentioned_terms" not in df.columns:
        return df
    wc_terms = []
    wc_feats = []
    for _, row in df.iterrows():
        r_coded = int(row["range"]) if "range" in df.columns and pd.notna(row["range"]) else 0
        t_str, c_str = extract_wordcloud_columns(row["mentioned_terms"], exp_type, r_coded)
        wc_terms.append(t_str)
        wc_feats.append(c_str)
    df["wordcloud_terms"] = wc_terms
    df["wordcloud_features"] = wc_feats
    return df


In [42]:
# ------------------------------------------------------------
# 5. Load data and run batch coding across all 6 experiments
# ------------------------------------------------------------
# Set FORCE_RECODE = True to re-code all sheets from scratch (e.g., after updating the prompt);
# set False to resume and skip already-coded participants.
FORCE_RECODE = False

print(f"Active Prompt Style: [{PROMPT_STYLE}]")
print(f"Target Output File:  [{OUTPUT_FILE}]")
print(f"Force Recode:        [{FORCE_RECODE}]\n")

# Load all existing sheets from OUTPUT_FILE if it exists (for resumability when FORCE_RECODE is False)
existing_sheets = {}
if not FORCE_RECODE and os.path.exists(OUTPUT_FILE):
    try:
        xl_existing = pd.ExcelFile(OUTPUT_FILE)
        for sheet_name in xl_existing.sheet_names:
            existing_sheets[sheet_name] = pd.read_excel(OUTPUT_FILE, sheet_name=sheet_name)
        print(f"Existing output workbook found with sheets: {list(existing_sheets.keys())}")
    except Exception as e:
        print(f"Could not read existing output workbook: {e}")

xl_input = pd.ExcelFile(INPUT_FILE)
print(f"Sheets found in input file: {xl_input.sheet_names}")
sheet_lookup = {s.strip().lower(): s for s in xl_input.sheet_names}

for sheet_name, cfg in EXPERIMENT_CONFIGS.items():
    print("\n" + "=" * 60)
    print(f"PROCESSING EXPERIMENT: {sheet_name}")
    print("=" * 60)

    actual_input_sheet = sheet_lookup.get(sheet_name.strip().lower())
    if actual_input_sheet is None:
        print(f"Sheet '{sheet_name}' not found in input file! (Available sheets: {xl_input.sheet_names}) Skipping.")
        continue

    exp_type = cfg["exp_type"]
    text_col = cfg["text_col"]
    feature_cols = cfg["feature_cols"]
    prompt = get_prompt(exp_type)

    df_raw = pd.read_excel(INPUT_FILE, sheet_name=actual_input_sheet)
    df_raw.columns = df_raw.columns.str.strip()

    # Identify text column fallback if needed
    if text_col not in df_raw.columns:
        if "open_answer" in df_raw.columns:
            text_col = "open_answer"
        elif "strategy_open" in df_raw.columns:
            text_col = "strategy_open"
        else:
            raise ValueError(f"Could not find response text column in {sheet_name}")

    # Clean participant IDs and text
    df_raw["subj_id"] = df_raw["subj_id"].astype(str).str.strip()
    df_raw[text_col] = df_raw[text_col].fillna("").astype(str).str.strip()

    print(f"Total rows in sheet: {len(df_raw)}")

    # Retrieve or initialize sheet dataframe
    if sheet_name in existing_sheets and "subj_id" in existing_sheets[sheet_name].columns:
        out = existing_sheets[sheet_name]
        out["subj_id"] = out["subj_id"].astype(str).str.strip()
        print(f"Found existing output for '{sheet_name}': {len(out)} rows")
    else:
        out = pd.DataFrame(columns=[
            "subj_id", text_col, *feature_cols,
            "mentioned_terms", "rationale", "uncertain"
        ])

    # Check which participants already have valid feature codes
    complete_ids = set()
    if not out.empty and all(col in out.columns for col in feature_cols):
        complete_ids = set(out.dropna(subset=feature_cols)["subj_id"].astype(str))

    todo = df_raw[~df_raw["subj_id"].isin(complete_ids)].copy()
    print(f"Already coded: {len(complete_ids)} | Needing coding: {len(todo)}")

    if len(todo) == 0:
        print(f"All participants in '{sheet_name}' already coded.")
        out = add_wordcloud_columns_to_df(out, exp_type)
        col_order = ["subj_id", text_col, *feature_cols, "mentioned_terms", "wordcloud_terms", "wordcloud_features", "rationale", "uncertain"]
        out = out.reindex(columns=[c for c in col_order if c in out.columns] + [c for c in out.columns if c not in col_order])
        existing_sheets[sheet_name] = out
        continue

    # Process batches
    for start in range(0, len(todo), BATCH_SIZE):
        batch = todo.iloc[start:start + BATCH_SIZE]
        batch_num = start // BATCH_SIZE + 1
        total_batches = (len(todo) - 1) // BATCH_SIZE + 1

        print(
            f"\n[{sheet_name}] Batch {batch_num}/{total_batches} "
            f"({len(batch)} responses)...",
            flush=True
        )

        try:
            batch_coded = code_batch(batch, text_col, feature_cols, prompt)
        except Exception as e:
            print(f"Batch failed: {e}", flush=True)
            batch_coded = [
                {
                    "subj_id": str(row["subj_id"]),
                    text_col: row[text_col],
                    **make_error_row(feature_cols, str(e))
                }
                for _, row in batch.iterrows()
            ]

        # Remove previous incomplete rows for these IDs and append new
        batch_ids = {str(x) for x in batch["subj_id"]}
        if not out.empty and "subj_id" in out.columns:
            out = out[~out["subj_id"].astype(str).isin(batch_ids)]

        out = pd.concat([out, pd.DataFrame(batch_coded)], ignore_index=True)
        out = add_wordcloud_columns_to_df(out, exp_type)

        # Standardize column order matching OOO analysis structure
        col_order = ["subj_id", text_col, *feature_cols, "mentioned_terms", "wordcloud_terms", "wordcloud_features", "rationale", "uncertain"]
        out = out.reindex(columns=[c for c in col_order if c in out.columns] + [c for c in out.columns if c not in col_order])

        existing_sheets[sheet_name] = out

        # Save all sheets to OUTPUT_FILE after every batch
        with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
            for s_name, s_df in existing_sheets.items():
                s_df.to_excel(writer, sheet_name=s_name, index=False)

        print(f"Saved progress to '{OUTPUT_FILE}' ({sheet_name}: {len(out)} rows).")
        time.sleep(2)

print("\nAll experiments processed.")


Active Prompt Style: [minimal]
Target Output File:  [/Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature_Open_Answers_LLM_coded_minimal.xlsx]
Force Recode:        [False]

Existing output workbook found with sheets: ['exp1 curv', 'exp1 noInstr', 'exp2 curv', 'exp2 noInstr', 'exp3 curv', 'exp3 noInstr']
Sheets found in input file: ['exp1 curv', 'exp1 noInstr', 'exp2 curv', 'exp2 noInstr', 'exp3 curv', 'exp3 noInstr']

PROCESSING EXPERIMENT: exp1 curv
Total rows in sheet: 35
Found existing output for 'exp1 curv': 35 rows
Already coded: 35 | Needing coding: 0
All participants in 'exp1 curv' already coded.

PROCESSING EXPERIMENT: exp1 noInstr
Total rows in sheet: 36
Found existing output for 'exp1 noInstr': 36 rows
Already coded: 36 | Needing coding: 0
All participants in 'exp1 noInstr' already coded.

PROCESSING EXPERIMENT: exp2 curv
Total rows in sheet: 36
Found existing output for 'exp2 curv': 36 rows
Already coded: 36 | Needing coding: 0
All participants

In [43]:
# ------------------------------------------------------------
# 6. Brief summary & word-cloud column refresh across all experiments
# ------------------------------------------------------------
# Load all sheets into memory BEFORE any processing (never read inside ExcelWriter!)
sheets_to_summarize = {}
if os.path.exists(OUTPUT_FILE):
    try:
        sheets_to_summarize = pd.read_excel(OUTPUT_FILE, sheet_name=None, engine="openpyxl")
    except Exception as e:
        print(f"Could not read {OUTPUT_FILE} from disk ({e}). Using in-memory data if available.")
        if "existing_sheets" in globals():
            sheets_to_summarize = existing_sheets
elif "existing_sheets" in globals() and existing_sheets:
    sheets_to_summarize = existing_sheets

if sheets_to_summarize:
    with pd.ExcelWriter(OUTPUT_FILE, engine="openpyxl") as writer:
        for s_name, s_df in sheets_to_summarize.items():
            cfg = EXPERIMENT_CONFIGS.get(s_name)
            if cfg:
                for col in cfg["feature_cols"]:
                    if col in s_df.columns:
                        s_df[col] = pd.to_numeric(s_df[col], errors="coerce").astype("Int64")
                s_df = add_wordcloud_columns_to_df(s_df, cfg["exp_type"])
                text_c = cfg["text_col"] if cfg["text_col"] in s_df.columns else ("open_answer" if "open_answer" in s_df.columns else "strategy_open")
                col_order = ["subj_id", text_c, *cfg["feature_cols"], "mentioned_terms", "wordcloud_terms", "wordcloud_features", "rationale", "uncertain"]
                s_df = s_df.reindex(columns=[c for c in col_order if c in s_df.columns] + [c for c in s_df.columns if c not in col_order])
                sheets_to_summarize[s_name] = s_df
            s_df.to_excel(writer, sheet_name=s_name, index=False)
    print(f"Saved clean workbook (with wordcloud_terms & wordcloud_features): {OUTPUT_FILE}\n")

    summary_rows = []
    for sheet_name, df_sheet in sheets_to_summarize.items():
        cfg = EXPERIMENT_CONFIGS.get(sheet_name, {})
        feat_cols = [c for c in cfg.get("feature_cols", []) if c in df_sheet.columns]
        total_rows = len(df_sheet)
        coded_rows = int(df_sheet.dropna(subset=feat_cols).shape[0]) if feat_cols else 0
        errors = int(df_sheet["rationale"].astype(str).str.startswith("API_ERROR").sum()) if "rationale" in df_sheet.columns else 0
        uncertain = int(pd.to_numeric(df_sheet["uncertain"], errors="coerce").fillna(0).sum()) if "uncertain" in df_sheet.columns else 0
        
        summary_rows.append({
            "Experiment": sheet_name,
            "Status": "✓ Complete" if (coded_rows == total_rows and errors == 0) else "⚠ Check",
            "Total Rows": total_rows,
            "Coded": coded_rows,
            "API Errors": errors,
            "Uncertain": uncertain
        })
    
    summary_df = pd.DataFrame(summary_rows)
    print("=== Brief Experiment Summary ===")
    print(summary_df.to_string(index=False))
else:
    print(f"No valid output data found for {OUTPUT_FILE}.")


Saved clean workbook: /Users/seoheehan/Documents/GitHub/3_Curvature-Representation/Open_Answers/Curvature_Open_Answers_LLM_coded_minimal.xlsx

=== Brief Experiment Summary ===
  Experiment     Status  Total Rows  Coded  API Errors  Uncertain
   exp1 curv ✓ Complete          35     35           0          2
exp1 noInstr ✓ Complete          36     36           0          2
   exp2 curv ✓ Complete          36     36           0          6
exp2 noInstr ✓ Complete          36     36           0          0
   exp3 curv ✓ Complete          36     36           0          9
exp3 noInstr ✓ Complete          36     36           0          0
